In [ ]:
--- Day 17: Chronospatial Computer ---
The Historians push the button on their strange device, but this time, you all just feel like you're falling.

"Situation critical", the device announces in a familiar voice. "Bootstrapping process failed. Initializing debugger...."

The small handheld device suddenly unfolds into an entire computer! The Historians look around nervously before one of them tosses it to you.

This seems to be a 3-bit computer: its program is a list of 3-bit numbers (0 through 7), like 0,1,2,3. The computer also has three registers named A, B, and C, but these registers aren't limited to 3 bits and can instead hold any integer.

The computer knows eight instructions, each identified by a 3-bit number (called the instruction's opcode). Each instruction also reads the 3-bit number after it as an input; this is called its operand.

A number called the instruction pointer identifies the position in the program from which the next opcode will be read; it starts at 0, pointing at the first 3-bit number in the program. Except for jump instructions, the instruction pointer increases by 2 after each instruction is processed (to move past the instruction's opcode and its operand). If the computer tries to read an opcode past the end of the program, it instead halts.

So, the program 0,1,2,3 would run the instruction whose opcode is 0 and pass it the operand 1, then run the instruction having opcode 2 and pass it the operand 3, then halt.

There are two types of operands; each instruction specifies the type of its operand. The value of a literal operand is the operand itself. For example, the value of the literal operand 7 is the number 7. The value of a combo operand can be found as follows:

Combo operands 0 through 3 represent literal values 0 through 3.
Combo operand 4 represents the value of register A.
Combo operand 5 represents the value of register B.
Combo operand 6 represents the value of register C.
Combo operand 7 is reserved and will not appear in valid programs.
The eight instructions are as follows:

The adv instruction (opcode 0) performs division. The numerator is the value in the A register. The denominator is found by raising 2 to the power of the instruction's combo operand. (So, an operand of 2 would divide A by 4 (2^2); an operand of 5 would divide A by 2^B.) The result of the division operation is truncated to an integer and then written to the A register.

The bxl instruction (opcode 1) calculates the bitwise XOR of register B and the instruction's literal operand, then stores the result in register B.

The bst instruction (opcode 2) calculates the value of its combo operand modulo 8 (thereby keeping only its lowest 3 bits), then writes that value to the B register.

The jnz instruction (opcode 3) does nothing if the A register is 0. However, if the A register is not zero, it jumps by setting the instruction pointer to the value of its literal operand; if this instruction jumps, the instruction pointer is not increased by 2 after this instruction.

The bxc instruction (opcode 4) calculates the bitwise XOR of register B and register C, then stores the result in register B. (For legacy reasons, this instruction reads an operand but ignores it.)

The out instruction (opcode 5) calculates the value of its combo operand modulo 8, then outputs that value. (If a program outputs multiple values, they are separated by commas.)

The bdv instruction (opcode 6) works exactly like the adv instruction except that the result is stored in the B register. (The numerator is still read from the A register.)

The cdv instruction (opcode 7) works exactly like the adv instruction except that the result is stored in the C register. (The numerator is still read from the A register.)

Here are some examples of instruction operation:

If register C contains 9, the program 2,6 would set register B to 1.
If register A contains 10, the program 5,0,5,1,5,4 would output 0,1,2.
If register A contains 2024, the program 0,1,5,4,3,0 would output 4,2,5,6,7,7,7,7,3,1,0 and leave 0 in register A.
If register B contains 29, the program 1,7 would set register B to 26.
If register B contains 2024 and register C contains 43690, the program 4,0 would set register B to 44354.
The Historians' strange device has finished initializing its debugger and is displaying some information about the program it is trying to run (your puzzle input). For example:

Register A: 729
Register B: 0
Register C: 0

Program: 0,1,5,4,3,0
Your first task is to determine what the program is trying to output. To do this, initialize the registers to the given values, then run the given program, collecting any output produced by out instructions. (Always join the values produced by out instructions with commas.) After the above program halts, its final output will be 4,6,3,5,6,3,5,2,1,0.

Using the information provided by the debugger, initialize the registers to the given values, then run the program. Once it halts, what do you get if you use commas to join the values it output into a single string?

Your puzzle answer was 4,1,5,3,1,5,3,5,7.

--- Part Two ---
Digging deeper in the device's manual, you discover the problem: this program is supposed to output another copy of the program! Unfortunately, the value in register A seems to have been corrupted. You'll need to find a new value to which you can initialize register A so that the program's output instructions produce an exact copy of the program itself.

For example:

Register A: 2024
Register B: 0
Register C: 0

Program: 0,3,5,4,3,0
This program outputs a copy of itself if register A is instead initialized to 117440. (The original initial value of register A, 2024, is ignored.)

What is the lowest positive initial value for register A that causes the program to output a copy of itself?

Your puzzle answer was 164542125272765.

# Part 1

In [105]:
def read_input(p_filename):

    # Sample input
    '''    
    Register A: 729
    Register B: 0
    Register C: 0
     
    Program: 0,1,5,4,3,0
    '''

    v_program = dict()
    
    with open(p_filename, 'r') as file:
        for row in file:     
            
            try:
                (key, val) = (row.strip().split(':'))
            except:
                pass # There is one empty row to be skipped

            val = val.strip()          
            v_program[key] = val

            if v_program[key].isdigit(): # Take registers
                v_program[key] = int(v_program[key])

    v_instructions = list()
    opcode = None
    operand = None
    for i in v_program['Program'].split(','):
        if opcode is None: # Found first value of a pair
            opcode = i 
        else: # Found second value of a pair
            operand = i
        
        if opcode and operand: # The pair is found - write it and clean up for the next pair
            v_instructions.append((int(opcode), int(operand)))
            opcode = None
            operand = None   

    v_program['Program'] = v_instructions
    
    return v_program

In [106]:
def display_program():
    v_str = ''
    for (x, y) in program['Program']:
        v_str += ',' + str(x)
        v_str += ',' + str(y)

    v_str = v_str.strip(',')
    return v_str

In [107]:
def get_combo(p_value):
    
    """
    There are two types of operands; each instruction specifies the type of its operand. The value of a literal operand is the operand itself. For example, the value of the literal operand 7 is the number 7. The value of a combo operand can be found as follows:
    
    Combo operands 0 through 3 represent literal values 0 through 3.
    Combo operand 4 represents the value of register A.
    Combo operand 5 represents the value of register B.
    Combo operand 6 represents the value of register C.
    Combo operand 7 is reserved and will not appear in valid programs.
    """
    
    if p_value <= 3:
        return p_value
    elif p_value == 4:
        return program['Register A']
    elif p_value == 5:
        return program['Register B']
    elif p_value == 6:
        return program['Register C']

In [108]:
def adv():
    '''
The adv instruction (opcode 0) performs division. The numerator is the value in the A register. 
The denominator is found by raising 2 to the power of the instruction's combo operand. 
(So, an operand of 2 would divide A by 4 (2^2); an operand of 5 would divide A by 2^B.) 
The result of the division operation is truncated to an integer and then written to the A register.
    '''
    
    pointer = program['Instruction pointer']
    numerator = program['Register A']
    denominator = get_combo(program['Program'][pointer][1])
    result = numerator // (2**denominator)
    program['Register A'] = result
    program['Instruction pointer'] += 1

In [109]:
def bxl():
    '''
The bxl instruction (opcode 1) calculates the bitwise XOR of register B and the instruction's literal operand, then stores the result in register B.
    '''
    
    pointer = program['Instruction pointer']    
    operand1 = program['Program'][pointer][1] # Get the operand in the pair
    operand2 = program['Register B']
    result = operand1 ^ operand2
    program['Register B'] = result
    program['Instruction pointer'] += 1

In [110]:
def bst(debug=False):
    '''
The bst instruction (opcode 2) calculates the value of its combo operand modulo 8 (thereby keeping only its lowest 3 bits), 
then writes that value to the B register.    
    '''
    
    pointer = program['Instruction pointer']     
    operator1 = get_combo(program['Program'][pointer][1])
    result = operator1 % 8
    if debug:
        print(f'Executed 2_bst instruction with operator1 = {operator1} and result = {result}')
    program['Register B'] = result
    program['Instruction pointer'] += 1

In [111]:
def jnz():
    '''
The jnz instruction (opcode 3) does nothing if the A register is 0. 
However, if the A register is not zero, it jumps by setting the instruction pointer to the value of its literal operand; 
if this instruction jumps, the instruction pointer is not increased by 2 after this instruction.
    '''
    
    pointer = program['Instruction pointer']    
    operand1 = program['Register A']
    operand2 = program['Program'][pointer][1] # Get the operand in the pair
    if operand1 == 0:
        program['Instruction pointer'] += 1
    else:
        program['Instruction pointer'] = operand2 // 2

In [112]:
def bxc():
    '''
The bxc instruction (opcode 4) calculates the bitwise XOR of register B and register C, then stores the result in register B. 
(For legacy reasons, this instruction reads an operand but ignores it.)
    '''
    
    pointer = program['Instruction pointer']    
    operand1 = program['Register B']
    operand2 = program['Register C']
    result = operand1 ^ operand2
    program['Register B'] = result
    program['Instruction pointer'] += 1

In [113]:
def out():
    '''
The out instruction (opcode 5) calculates the value of its combo operand modulo 8, then outputs that value. 
(If a program outputs multiple values, they are separated by commas.)
    '''
    
    pointer = program['Instruction pointer']    
    operator1 = get_combo(program['Program'][pointer][1])
    program['Output'] += ',' + str(operator1 % 8)
    program['Output'] = program['Output'].strip(',')
    program['Instruction pointer'] += 1

In [114]:
def bdv():
    '''
The bdv instruction (opcode 6) works exactly like the adv instruction except that the result is stored in the B register. 
(The numerator is still read from the A register.)
    '''
    
    pointer = program['Instruction pointer']
    numerator = program['Register A']
    denominator = get_combo(program['Program'][pointer][1])
    result = numerator // (2**denominator)
    program['Register B'] = result
    program['Instruction pointer'] += 1

In [115]:
def cdv():
    '''
The cdv instruction (opcode 7) works exactly like the adv instruction except that the result is stored in the C register. 
(The numerator is still read from the A register.)
    '''
    
    pointer = program['Instruction pointer']
    numerator = program['Register A']
    denominator = get_combo(program['Program'][pointer][1])
    result = numerator // (2**denominator)
    program['Register C'] = result
    program['Instruction pointer'] += 1

In [116]:
def make_one_itteration(debug=False):
    v_pointer = program['Instruction pointer']
    (v_opcode, v_operand) = program['Program'][v_pointer]
    if debug:
        print(f'Execution of make_one_itteration for (v_opcode, v_operand) =  ({v_opcode}, {v_operand})')
    if v_opcode == 0:
        adv()
    if v_opcode == 1:
        bxl()    
    if v_opcode == 2:
        bst(debug)
    if v_opcode == 3:
        jnz()
    if v_opcode == 4:
        bxc()    
    if v_opcode == 5:
        out()   
    if v_opcode == 6:
        bdv()   
    if v_opcode == 7:
        cdv()   

In [117]:
def execute_program():
    program_length = len(program['Program'])
    while program['Instruction pointer'] < program_length:
        make_one_itteration()
    program['Output'] = program['Output'].strip(',')

In [120]:
input_file_name = 'input.txt'

program = read_input(input_file_name)
program['Instruction pointer'] = 0
program['Output'] = ''

print(program)

v_display = display_program()
v_register_a = 0
execute_program()
v_output = program['Output']

print (v_output)

{'Register A': 56256477, 'Register B': 0, 'Register C': 0, 'Program': [(2, 4), (1, 1), (7, 5), (1, 5), (0, 3), (4, 3), (5, 5), (3, 0)], 'Instruction pointer': 0, 'Output': ''}
4,1,5,3,1,5,3,5,7


In [121]:
print(''.join(program['Output']))
print(display_program())

4,1,5,3,1,5,3,5,7
2,4,1,1,7,5,1,5,0,3,4,3,5,5,3,0


# Part 2

In [139]:
def is_self_program(program, debug=False):

    '''
Too slow
    '''
    
    
    program_length = len(program['Program'])
    v_prg = display_program()
    v_reg_a = program['Register A']
    should_continue = True

    if debug:
        print(f"Calculating is_self_program for program={program}, display_program='{v_prg}', program_length={program_length}")
    
    while program['Instruction pointer'] < program_length and should_continue:
        make_one_itteration(debug)
        v_opt = program['Output'].strip(',') 
        if debug:
            print(f"After execution of one itteration: program={program}, display_program='{v_prg}', output={v_opt}")
        if not v_prg.startswith(v_opt):    
            if debug:
                print ('No match')
            should_continue = False
            return False

    #print(f'For {v_reg_a} check if v_prg = {v_prg} starts with {v_opt}')
    
    if program['Output'].strip(',') == v_prg:
        return True
    else:
        return False

In [ ]:
def calc_one_output(program, debug=False):
    

In [140]:
input_file_name = 'input.txt'

program = read_input(input_file_name)
program['Instruction pointer'] = 0
program['Output'] = ''

print(f"Start with {program}")

Start with {'Register A': 56256477, 'Register B': 0, 'Register C': 0, 'Program': [(2, 4), (1, 1), (7, 5), (1, 5), (0, 3), (4, 3), (5, 5), (3, 0)], 'Instruction pointer': 0, 'Output': ''}


In [141]:
program['Register A'] = 164542125272765
program['Instruction pointer'] = 0

print(is_self_program(program, debug=False))

True


In [158]:
def func(A):    
    #B = ((((A % 8) ^ 1) ^ 5) ^ ( A // (2** ((A % 8) ^ 1))))
    B = ((((A & 7) ^ 1) ^ 5) ^ ( A // (2** ((A & 7) ^ 1))))
    #C = A // (2** ((A % 8) ^ 1))
    #A = A // (2**3)
    A = A // 8
    return((A, B%8))

In [152]:
def func_pr(A, program):    
    program['Register A'] = A    
    program['Program'] = A   
    for x in range(len(program['Program'])):
        make_one_itteration()
    return((program['Register A'], program['Register B']%8))

In [159]:
def find_solution(A, p_target_list):
    x = 0
    if len(p_target_list) < 1:
        return ''
    while x < 8:
        (A1, B) = func(A + x)
        print(f'B={B} for A={A}, x={x} and p_target_list={p_target_list}')
        if B == p_target_list[-1]:
            
            next_part = find_solution((A+x)*8, p_target_list[:-1])
            
            if next_part and len(p_target_list) > 1:
                print(f'B={B} for A={A+x} and p_target_list={p_target_list}')
                return str(B) + next_part
            elif len(p_target_list) == 1:
                print(f'B={B} for A={A+x}')
                return str(B)
        x += 1
    return ''

In [160]:
digit_list = list((2, 4, 1, 1, 7, 5, 1, 5, 0, 3, 4, 3, 5, 5, 3, 0))
#digit_list.reverse()
solution = find_solution(0, digit_list)

B=4 for A=0, x=0 and p_target_list=[2, 4, 1, 1, 7, 5, 1, 5, 0, 3, 4, 3, 5, 5, 3, 0]
B=4 for A=0, x=1 and p_target_list=[2, 4, 1, 1, 7, 5, 1, 5, 0, 3, 4, 3, 5, 5, 3, 0]
B=6 for A=0, x=2 and p_target_list=[2, 4, 1, 1, 7, 5, 1, 5, 0, 3, 4, 3, 5, 5, 3, 0]
B=7 for A=0, x=3 and p_target_list=[2, 4, 1, 1, 7, 5, 1, 5, 0, 3, 4, 3, 5, 5, 3, 0]
B=0 for A=0, x=4 and p_target_list=[2, 4, 1, 1, 7, 5, 1, 5, 0, 3, 4, 3, 5, 5, 3, 0]
B=4 for A=32, x=0 and p_target_list=[2, 4, 1, 1, 7, 5, 1, 5, 0, 3, 4, 3, 5, 5, 3]
B=4 for A=32, x=1 and p_target_list=[2, 4, 1, 1, 7, 5, 1, 5, 0, 3, 4, 3, 5, 5, 3]
B=2 for A=32, x=2 and p_target_list=[2, 4, 1, 1, 7, 5, 1, 5, 0, 3, 4, 3, 5, 5, 3]
B=7 for A=32, x=3 and p_target_list=[2, 4, 1, 1, 7, 5, 1, 5, 0, 3, 4, 3, 5, 5, 3]
B=1 for A=32, x=4 and p_target_list=[2, 4, 1, 1, 7, 5, 1, 5, 0, 3, 4, 3, 5, 5, 3]
B=3 for A=32, x=5 and p_target_list=[2, 4, 1, 1, 7, 5, 1, 5, 0, 3, 4, 3, 5, 5, 3]
B=0 for A=296, x=0 and p_target_list=[2, 4, 1, 1, 7, 5, 1, 5, 0, 3, 4, 3, 5, 5]
B=4 for 